In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from dotenv.main import load_dotenv

load_dotenv()
REPO_PATH = os.getenv('REPO_PATH')

sys.path.append(REPO_PATH)

In [2]:
import json

def load_json(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)

def save_json(file_name, file):
    with open(os.path.join(REPO_PATH, "data_files", f"{file_name}.json"), "w", encoding="utf-8") as out_file:
        json.dump(file, out_file, indent=4)

# TODO:
1. Wikipedia harvest - Run as is for now, try to understand why it crushes near the end.
2. Extract tables:
    - length between 3 to 10
    - table is square and parsed normally (no merged cells)
    - full table
    - remove some columns with no relevant data (empty or by keywords)
3. Add to table extraction text titles and relevant paragraphs for the context.
4. Create groups - all the table, try to see what is the main entity - return the column name, if they don't have one, discard the table and return 'None'.
5. Create Questions
    - strategy injection, sample a subset of strategies in a shuffled order, if not possible than discard table.
    - make a function to extract the answer per question strategy --> create GT answer.
    - Add the 'group size' strategy.
    - constract the data samples.

### Havest Wikipedia pages with tables:

In [53]:
from knowledge_maze.create_dataset.create_questions.wikipedia_mining import WikiHarvester
# Initialize Wikipedia harvester
harvester = WikiHarvester(delay=0.05)
# Run Phase 1 - Harvesting titles based on keywords
raw_titles = harvester.fetch_titles()
# Run Phase 2 - Filtering pages for tables
# table_titles = harvester.filter_pages_with_tables(raw_titles)

Starting Phase 1: Harvesting titles for 18 keywords...


RUNNING: 100%|███████████████████████████| 18/18 [03:44<00:00, 12.49s/keyword, current_kw=missions, total_pages=316557]


Keyword              | Pages Found
----------------------------------
list of              | 306682    
comparison of        | 1401      
rankings of          | 32        
discography of       | 131       
filmography of       | 191       
demographics of      | 1730      
members              | 2915      
teams                | 44        
countries            | 321       
cities               | 674       
species              | 234       
films                | 282       
awards               | 1177      
companies            | 197       
airlines             | 78        
universities         | 338       
planets              | 80        
missions             | 50        
----------------------------------
TOTAL UNIQUE         | 316557    



In [38]:
from knowledge_maze.create_dataset.create_questions.wikipedia_mining import WikiHarvester
# Initialize Wikipedia harvester
harvester = WikiHarvester()
# Run Phase 1 - Harvesting titles based on keywords
harvester.run_complete_harvest()
# Run Phase 2 - Filtering pages for tables
harvester.filter_pages_with_tables()

RUNNING:   0%|          | 0/228 [00:00<?, ?keyword/s]


TOTAL UNIQUE TITLES COLLECTED: 113466



,Method,Keyword,Count
0,Prefix,List of cities,1537
1,Prefix,List of towns,501
2,Prefix,List of villages,483
3,Prefix,List of districts,375
4,Prefix,List of municipalities,1021
5,Prefix,List of communes,194
6,Prefix,List of counties,259
7,Prefix,List of provinces,84
8,Prefix,List of islands,629
9,Prefix,List of mountains,571


FILTERING:   0%|          | 0/113466 [00:00<?, ?page/s]

KeyboardInterrupt: 

In [47]:
await harvester.filter_pages_async()

ASYNC FILTER:   0%|          | 0/113466 [00:00<?, ?page/s]

⚠️ 429 Throttled: Sleeping 5s (Attempt 1/3)
⚠️ 429 Throttled: Sleeping 5s (Attempt 1/3)
⚠️ 429 Throttled: Sleeping 5s (Attempt 1/3)
⚠️ 429 Throttled: Sleeping 5s (Attempt 1/3)
⚠️ 429 Throttled: Sleeping 5s (Attempt 1/3)
⚠️ 429 Throttled: Sleeping 5s (Attempt 1/3)
⚠️ 429 Throttled: Sleeping 5s (Attempt 1/3)
⚠️ 429 Throttled: Sleeping 5s (Attempt 1/3)
⚠️ 429 Throttled: Sleeping 5s (Attempt 1/3)
⚠️ 429 Throttled: Sleeping 5s (Attempt 1/3)
⚠️ 429 Throttled: Sleeping 5s (Attempt 1/3)
⚠️ 429 Throttled: Sleeping 5s (Attempt 1/3)
⚠️ 429 Throttled: Sleeping 5s (Attempt 1/3)
⚠️ 429 Throttled: Sleeping 10s (Attempt 2/3)
⚠️ 429 Throttled: Sleeping 10s (Attempt 2/3)
⚠️ 429 Throttled: Sleeping 10s (Attempt 2/3)
⚠️ 429 Throttled: Sleeping 5s (Attempt 1/3)
⚠️ 429 Throttled: Sleeping 10s (Attempt 2/3)
⚠️ 429 Throttled: Sleeping 10s (Attempt 2/3)
⚠️ 429 Throttled: Sleeping 10s (Attempt 2/3)
⚠️ 429 Throttled: Sleeping 10s (Attempt 2/3)
⚠️ 429 Throttled: Sleeping 10s (Attempt 2/3)
⚠️ 429 Throttled: Sleepi

In [48]:
len(harvester.valid_titles)

39713

In [49]:
from knowledge_maze.create_dataset.create_questions.table_processing import WikiExtractor
runner = WikiExtractor(max_workers=25)
final_data_aaaaaa = runner.bulk_extract(random.sample(harvester.valid_titles, 1000))

Starting Extraction: 1000 titles | 25 concurrent threads.


Extracting Tables: 100%|████████████████████| 1000/1000 [16:01<00:00,  1.04page/s, Status=RUNNING, Success=685, 429s=0]


FINAL EXTRACTION SUMMARY
------------------------------------------------------------
Total Titles Input:                                | 1000
Pages with Successfully Saved Tables:              | 685
Pages with No Tables Found / Filtered Out:         | 311
Rejected due to Rate Limit (429):                  | 0
Other Network Errors:                              | 2
------------------------------------------------------------
TOTAL TABLES COLLECTED:                            | 2068
SUCCESS RATE:                                      | 68.5%



In [57]:
import pandas as pd

short = 0 #[0, 0, 0, 0, 0, 0, 0, 0]
long = 0
for l in [len(sample['rows']) for sample in final_data_aaaaaa]:
    if l > 12:
        long += 1
    else:
        short += 1
        # short[l - 3] += 1

print(f"Short: {short}, Long: {long}")
print("=" * 100)

for i in range(3, 13):
    for t in final_data_aaaaaa:
        if len(t['rows']) == i:
            print(t['table_name'])
            tttt = pd.DataFrame(t['rows'])
            display(tttt)
            print("="*100)
            break

Short: 1344, Long: 724
Now That's What I Call Country > Chart performance > Year-end charts


,Chart (2008),Position
0,US Top Country Albums (Billboard),35
1,Chart (2009),Position
2,US Top Country Albums (Billboard),34


Aotearoa Music Award for Best Group > Recipients > Recording Arts Talent Awards: Recording Artist/Group of the Year (1973 to 1977)


,Year,Winner
0,1973,Shona Laing
1,1974,Bulldogs Allstar Goodtime Band
2,1975,Mark Williams
3,1976,Dr Tree


MC Lars discography > Collaborative releases


...

,Category,Author
0,Lifetime Achievement,Daw Yin Yin (Saw Monnyin)
1,Novel,Hsinbyukyun Aung Thein
2,Collected short stories,Yin Yin Nu (Mandalay)
3,Belles-letters,Maung Moe Yan
4,Myanmar culture and fine arts literature,Minshin Aung (Twantay)
5,Child literature,Tin Lin Zaw
6,Translation,Nan Mya Kay Khine
7,General knowledge (arts),Ma Kyan
8,General knowledge (science),Aye Kyaw
9,Political,Pe Thein Tin (ambassador)


Myanmar National Literature Award > List > 2008


,Category,Author
0,Life-Time Achievement,Sayagyi Dr Kyaw Sein
1,Belle-letter,Yein Nwe Par
2,Selected short stories,Kyu Kyu Thin
3,Myanmar culture and fine arts literature,Myat Wai Toe (Institute of Education)
4,Child literature,Yenanmye Maung Manug Kyi
5,Youth literature,Kyunsinmye Saw Khet
6,Translation (general knowledge),Taw Kaung Min
7,Translation (aesthetics),Win Htut Zaw
8,General knowledge (arts),Dr Dagon
9,General knowledge (science),Dr Kyaw Tint


Bingham University > Academic programs


,Faculty,Undergraduate Programs,Postgraduate Programs
0,Faculty of Clinical Sciences,"MBBS Medicine, B.Sc. Nursing, B.Sc. Radiography","M.Sc. Public Health, Ph.D. Anatomy"
1,Faculty of Allied Health Sciences,"B.Sc. Anatomy, B.Sc. Physiology, B.Sc. Medical...","M.Sc. Physiology, Ph.D. Microbiology"
2,Faculty of Pharmaceutical Sciences,"B.Pharm. Pharmacy, B.Sc. Pharmacology",M.Sc. Pharmaceutical Microbiology
3,Faculty of Science and Technology,"B.Sc. Computer Science, B.Sc. Cyber Security, ...","M.Sc. Environmental Chemistry, Ph.D. Biochemistry"
4,Faculty of Law,LL.B. Bachelor of Law,None
5,Faculty of Environmental Sciences,"B.Sc. Architecture, B.Sc. Quantity Surveying, ...",M.Sc. Architecture
6,Faculty of Arts,"B.A. English, B.A. Philosophy, B.A. Religious ...","M.Sc. English & Literary Studies, Ph.D. English"
7,Faculty of Social Sciences,"B.Sc. Sociology, B.Sc. Political Science, B.Sc...","M.Sc. Mass Communication, Ph.D. Economics"
8,Faculty of Administration,"B.Sc. Accounting, B.Sc. Business Administration","M.Sc. Accounting, Ph.D. Business Administration"
9,Faculty of Education,"B.Sc. Guidance and Counseling, B.Sc. Psychology",None


In [8]:
save_json("table_titles", table_titles)

In [3]:
table_titles = load_json("C:/Users/Daniel/Desktop/KnowledgeMaze/data_files/table_titles.json")

### Parse and filter Wikipedia tables:

In [8]:
from knowledge_maze.create_dataset.create_questions.table_processing import WikiExtractor
import random
runner = WikiExtractor(max_workers=25)
final_data = runner.bulk_extract(random.sample(table_titles, 3000))

Starting Extraction: 3000 titles | 25 concurrent threads.


Extracting Tables: 100%|███████████████████| 3000/3000 [51:32<00:00,  1.03s/page, Status=RUNNING, Success=2114, 429s=0]


FINAL EXTRACTION SUMMARY
------------------------------------------------------------
Total Titles Input:                                | 3000
Pages with Successfully Saved Tables:              | 2114
Pages with No Tables Found / Filtered Out:         | 874
Rejected due to Rate Limit (429):                  | 0
Other Network Errors:                              | 7
------------------------------------------------------------
TOTAL TABLES COLLECTED:                            | 7104
SUCCESS RATE:                                      | 70.5%



In [6]:
final_data[0]

{'table_name': 'List of bean-to-bar chocolate manufacturers > Bean-to-bar chocolate manufacturers',
 'rows': [{'Company': 'Ah Cacao Real Chocolate',
   'Location': 'Mexico',
   'Year founded': 2003,
   'Products': 'Single origin chocolate bars',
   'Notes': 'Artisan producer of dark chocolate and related products made of Mexican cacao. Supports wildlife conservation.'},
  {'Company': 'Amano Artisan Chocolate',
   'Location': 'United States',
   'Year founded': 2006,
   'Products': 'Single origin chocolate (bars, couverture), cocoa nibs',
   'Notes': "Founding member of the Craft Chocolate Makers of America. The Madagascar bar and the Dos Rios Palet d'or have received gold medals from the London Academy of Chocolate."},
  {'Company': 'Amedei',
   'Location': 'Italy',
   'Year founded': 1990,
   'Products': '',
   'Notes': 'Amedei is a luxury chocolate manufacturing company located in Pontedera in the Tuscany region of Italy.'},
  {'Company': 'Askinosie Chocolate',
   'Location': 'United

### Testing:

In [36]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import get_group
try:
    import nest_asyncio
    nest_asyncio.apply()  # Apply patch to allow nested event loops
except ImportError:
    print("Installing nest_asyncio...")


llm_output = get_group(final_data[0])
llm_output

TableGroup(group_description='Bean-to-bar chocolate manufacturers founded 2010-2015', filter_column='Year founded', filter_type='numerical_range', filter_value=[2010, 2015])

In [37]:
import re
import traceback

def filter_table_rows_dict(llm_data, table_rows):
    try:
        # 1. Extract data using Pydantic V2 standards
        if hasattr(llm_data, 'model_dump'):
            data = llm_data.model_dump()
        elif hasattr(llm_data, 'Output') and hasattr(llm_data.Output, 'model_dump'):
            data = llm_data.Output.model_dump()
        elif hasattr(llm_data, 'dict'): # Fallback for V1
            data = llm_data.dict()
        else:
            data = llm_data

        col_name_query = str(data.get('filter_column', "")).lower().strip()
        f_type = str(data.get('filter_type', "")).lower()
        f_val = data.get('filter_value', [])

        if not col_name_query or not table_rows:
            return []

        # 2. Strict Column Matching
        # We look for the BEST match, not just any match
        target_col = None
        for k in table_rows[0].keys():
            # Clean both for a fair comparison
            clean_k = str(k).lower().strip()
            if col_name_query == clean_k or col_name_query in clean_k:
                target_col = k
                break
        
        if target_col is None:
            return []

        filtered_rows = []
        for row in table_rows:
            val_orig = row.get(target_col)
            if val_orig is None: continue
            
            # Sanitize cell content (Handles tuples/lists from scraper)
            if isinstance(val_orig, (tuple, list)) and len(val_orig) > 0:
                cell_str = str(val_orig[0])
            else:
                cell_str = str(val_orig)
            
            cell_str = cell_str.strip()
            if not cell_str: continue

            # --- Strict Filter Logic ---
            if 'list' in f_type:
                # Exact or substring match for list items
                low_cell = cell_str.lower()
                if any(str(item).lower() in low_cell for item in f_val):
                    filtered_rows.append(row)

            elif 'num' in f_type:
                try:
                    # Extract only the first number found (ignores citations like [1])
                    numbers = re.findall(r"[-+]?\d*\.\d+|\d+", cell_str.replace(',', ''))
                    if not numbers: continue
                    
                    val_as_float = float(numbers[0])
                    low_bound = float(f_val[0])
                    high_bound = float(f_val[1])
                    
                    # Core check
                    if low_bound <= val_as_float <= high_bound:
                        filtered_rows.append(row)
                except:
                    continue

            elif 'temp' in f_type:
                try:
                    # Look for 4-digit years
                    year_match = re.search(r'\b(1\d{3}|20\d{2})\b', cell_str)
                    if year_match:
                        row_year = int(year_match.group(0))
                        # Handle date strings or years in f_val
                        start_y = int(str(f_val[0])[:4])
                        end_y = int(str(f_val[1])[:4])
                        if start_y <= row_year <= end_y:
                            filtered_rows.append(row)
                except:
                    continue

        return filtered_rows

    except Exception as e:
        print(f"CRITICAL FILTER ERROR: {e}")
        traceback.print_exc()
        return []

In [39]:
# After your LLM call
filtered_data = filter_table_rows_dict(llm_output, final_data[0]['rows'])

In [42]:
import re

def extract_entity_list(llm_data, table_rows):
    try:
        # Pydantic V2 migration safe
        data = llm_data.model_dump() if hasattr(llm_data, 'model_dump') else llm_data
        
        ent_col_query = str(data.get('entity_column', "")).lower().strip()
        fil_col_query = str(data.get('filter_column', "")).lower().strip()
        f_type = data.get('filter_type')
        f_val = data.get('filter_value')
        sort_order = data.get('sort_order', 'desc')

        if not table_rows: return []

        # Find the actual column keys
        keys = table_rows[0].keys()
        entity_key = next((k for k in keys if ent_col_query in str(k).lower()), None)
        filter_key = next((k for k in keys if fil_col_query in str(k).lower()), None)

        if not entity_key or not filter_key: return []

        # --- STEP 1: Numerical Cleaning for Sorting/Filtering ---
        processed_data = []
        for row in table_rows:
            raw_ent = row.get(entity_key, "Unknown")
            raw_fil = row.get(filter_key, 0)
            
            # Extract name (handle tuples/lists from scraper)
            name = str(raw_ent[0] if isinstance(raw_ent, (list, tuple)) else raw_ent).strip()
            
            # Extract number for sorting/filtering
            clean_fil = re.sub(r'[^\d.]', '', str(raw_fil))
            val_as_float = float(clean_num[0]) if (clean_num := re.findall(r"[-+]?\d*\.\d+|\d+", str(raw_fil).replace(',', ''))) else 0.0
            
            processed_data.append({"name": name, "val": val_as_float, "original_row": row})

        # --- STEP 2: Apply Logic ---
        result_entities = []

        if f_type == "top_n":
            # Sort by the numerical value
            sorted_data = sorted(processed_data, key=lambda x: x['val'], reverse=(sort_order == 'desc'))
            # Take exactly N
            limit = min(max(int(f_val), 3), 10) # Safety clamp between 3 and 10
            result_entities = [d['name'] for d in sorted_data[:limit]]

        elif f_type == "numerical_range":
            # Traditional slice
            low, high = float(f_val[0]), float(f_val[1])
            matches = [d['name'] for d in processed_data if low <= d['val'] <= high]
            result_entities = matches[:10] # Ensure we don't exceed 10

        return result_entities

    except Exception as e:
        print(f"Extraction Error: {e}")
        return []

In [47]:
final_data[2]

{'table_name': 'List of Spanish films of 2017 > Box office',
 'rows': [{'Rank': 1,
   'Title': 'Tad the Lost Explorer and the Secret of King Midas (Tadeo Jones 2: El secreto del rey Midas)',
   'Distributor': 'Paramount Pictures',
   'Admissions': 3167110,
   'Domestic gross (€)': 17553149},
  {'Rank': 2,
   'Title': 'Perfect Strangers (Perfectos desconocidos)',
   'Distributor': 'Universal Pictures',
   'Admissions': 2020940,
   'Domestic gross (€)': 12969658},
  {'Rank': 3,
   'Title': "It's for Your Own Good (Es por tu bien)",
   'Distributor': 'Buena Vista International',
   'Admissions': 1584578,
   'Domestic gross (€)': 9638993},
  {'Rank': 4,
   'Title': 'Marrowbone (El secreto de Marrowbone)',
   'Distributor': 'Universal Pictures',
   'Admissions': 1158521,
   'Domestic gross (€)': 7153428},
  {'Rank': 5,
   'Title': 'Lord, Give Me Patience (Señor, dame paciencia)',
   'Distributor': 'Warner Bros. Pictures',
   'Admissions': 1106441,
   'Domestic gross (€)': 6682652},
  {'Rank

In [54]:
llm_output = get_group(final_data[0])
llm_output

TableGroup(group_description='Latest 7 chocolate manufacturers by founding date', entity_column='Company', filter_column='Year founded', filter_type='top_n', filter_value=7, sort_order='desc')

In [49]:
extract_entity_list(llm_output, final_data[2]['rows'])

['Tad the Lost Explorer and the Secret of King Midas (Tadeo Jones 2: El secreto del rey Midas)',
 'Perfect Strangers (Perfectos desconocidos)',
 "It's for Your Own Good (Es por tu bien)",
 'Marrowbone (El secreto de Marrowbone)',
 'Lord, Give Me Patience (Señor, dame paciencia)',
 'Toc Toc',
 'The Invisible Guardian (El guardián invisible)']

In [51]:
import re

def extract_grouped_data(llm_data, table_rows):
    try:
        # Pydantic V2 migration safe
        data = llm_data.model_dump() if hasattr(llm_data, 'model_dump') else llm_data
        
        group_desc = data.get('group_description', "")
        ent_col_query = str(data.get('entity_column', "")).lower().strip()
        fil_col_query = str(data.get('filter_column', "")).lower().strip()
        f_type = data.get('filter_type')
        f_val = data.get('filter_value')
        sort_order = data.get('sort_order', 'desc')

        if not table_rows: return {}

        # 1. Identify the actual keys in the original dictionary
        keys = list(table_rows[0].keys())
        entity_key = next((k for k in keys if ent_col_query in str(k).lower()), None)
        filter_key = next((k for k in keys if fil_col_query in str(k).lower()), None)

        if not entity_key or not filter_key: return {}

        # 2. Pre-process and Clean data
        processed_pool = []
        for row in table_rows:
            raw_ent = row.get(entity_key, "Unknown")
            raw_fil = row.get(filter_key, 0)
            
            # Clean entity name
            name = str(raw_ent[0] if isinstance(raw_ent, (list, tuple)) else raw_ent).strip()
            
            # Clean numerical value for sorting/filtering
            # Keep original raw_fil for the "Group entities" value output
            clean_num_list = re.findall(r"[-+]?\d*\.\d+|\d+", str(raw_fil).replace(',', ''))
            val_as_float = float(clean_num_list[0]) if clean_num_list else 0.0
            
            processed_pool.append({
                "name": name, 
                "numeric_val": val_as_float, 
                "display_val": raw_fil, # This is the "Y" in your entity:Y dict
                "original_row": row.copy() # Copy to avoid mutating original source
            })

        # 3. Filter/Sort Logic
        selected_matches = []
        if f_type == "top_n":
            sorted_pool = sorted(processed_pool, key=lambda x: x['numeric_val'], reverse=(sort_order == 'desc'))
            limit = min(max(int(f_val), 3), 10)
            selected_matches = sorted_pool[:limit]
        
        elif f_type == "numerical_range":
            low, high = float(f_val[0]), float(f_val[1])
            selected_matches = [d for d in processed_pool if low <= d['numeric_val'] <= high][:10]

        # 4. Construct Final Output Dict
        group_entities = {}
        group_data = []

        for match in selected_matches:
            # Add to entity:value dict
            group_entities[match['name']] = match['display_val']
            
            # Prepare row data: Remove the filter column
            pruned_row = match['original_row']
            if filter_key in pruned_row:
                del pruned_row[filter_key]
            
            group_data.append(pruned_row)

        return {
            "Group description": group_desc,
            "Group entities": group_entities,
            "Group data": group_data
        }

    except Exception as e:
        print(f"Extraction Error: {e}")
        return {}

In [55]:
extract_grouped_data(llm_output, final_data[0]['rows'])

{'Group description': 'Latest 7 chocolate manufacturers by founding date',
 'Group entities': {'Q sweet': 2019,
  'Fu Wan Chocolate': 2015,
  'Choco Togo': 2014,
  'Chocolala': 2013,
  'Castronovo Chocolate': 2012,
  'Malagos Chocolate': 2012,
  'Rick Jordan Chocolatier': 2011},
 'Group data': [{'Company': 'Q sweet',
   'Location': 'Taiwan',
   'Products': 'Chocolate truffles, bars, gift boxes',
   'Notes': ''},
  {'Company': 'Fu Wan Chocolate',
   'Location': 'Taiwan',
   'Products': 'Single-origin dark chocolates, Fermented and specialty bars combining cocoa with local flavors (e.g., lychee, tea, shrimp), Drinking chocolate, Limited editions based on Taiwanese terroir',
   'Notes': ''},
  {'Company': 'Choco Togo',
   'Location': 'Togo',
   'Products': 'Chocolate bars, hot chocolate, chocolate cake, and other chocolate products',
   'Notes': 'The first chocolate manufacturer in Togo.'},
  {'Company': 'Chocolala',
   'Location': 'Estonia',
   'Products': 'Single origin chocolate bars (

In [17]:
short = 0 #[0, 0, 0, 0, 0, 0, 0, 0]
long = 0
for l in [len(sample['rows']) for sample in final_data]:
    if l > 10:
        long += 1
    else:
        short += 1
        # short[l - 3] += 1

print(f"Short: {short}, Long: {long}")
print("=" * 100)

for i in range(3, 11):
    for t in final_data:
        if len(t['rows']) == i:
            print(t['table_name'])
            break

Short: 3942, Long: 3162
Comparison of OS emulation or virtualization apps on Android > Terminal emulation apps utilizing internal OS
ECHL awards > Team trophies
42nd Saturn Awards > Winners and nominees > Television > Programs
List of Renaissance band members > Members > Current
2011 ESPY Awards > Categories
List of Tennessee Titans head coaches > Key
List of Welsh flags > Regions, counties and cities > Traditional counties
ECHL awards > Individual trophies and awards


In [93]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import process_table_batches

# Assuming your tables are in a list of dicts
my_tables = random.sample(final_data, 10)

# Run the async loop
results = await process_table_batches(my_tables)

# Results will be a list of TableGroup objects/dicts
for my_tab, res in zip(my_tables, results):
    print(res.group_description)
    print(extract_grouped_data(res, my_tab['rows']))

Usage cost: 0.0032$ (Estimated: 0.00$): 100%|██████████████████████████████████████████| 10/10 [00:03<00:00,  3.30it/s]

The 5 sports with the highest total medal counts in mixed teams at the Olympics by Total medals.
{'Group description': 'The 5 sports with the highest total medal counts in mixed teams at the Olympics by Total medals.', 'Group entities': {'Total': 10, 'Tennis': 6, 'Tug of war': 1, 'Alpinism': 1, 'Polo': 1}, 'Group data': [{'Sport': 'Total', 'Gold': 4, 'Silver': 2, 'Bronze': 4}, {'Sport': 'Tennis', 'Gold': 1, 'Silver': 2, 'Bronze': 3}, {'Sport': 'Tug of war', 'Gold': 1, 'Silver': 0, 'Bronze': 0}, {'Sport': 'Alpinism', 'Gold': 1, 'Silver': 0, 'Bronze': 0}, {'Sport': 'Polo', 'Gold': 0, 'Silver': 0, 'Bronze': 1}]}
The 4 diplomatic missions of Indonesia in Oceania by host country name
{'Group description': 'The 4 diplomatic missions of Indonesia in Oceania by host country name', 'Group entities': {'Australia': 'Australia', 'Fiji': 'Fiji', 'New Zealand': 'New Zealand', 'Papua New Guinea': 'Papua New Guinea'}, 'Group data': [{'Host city': 'Canberra', 'Mission level': 'Embassy', 'Concurrent acc

In [94]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import extract_grouped_data

# Results will be a list of TableGroup objects/dicts
for my_tab, res in zip(my_tables, results):
    try:
        print(my_tab['table_name'])
        print(res.group_description)
        print(extract_grouped_data(res, my_tab['rows'])['Group entities'])
    except:
        pass

Mixed teams at the Olympics > Medal table > Medals by sport
The 5 sports with the highest total medal counts in mixed teams at the Olympics by Total medals.
{'Total': 10, 'Tennis': 6, 'Tug of war': 1, 'Alpinism': 1, 'Polo': 1}
List of diplomatic missions of Indonesia > Current missions > Oceania
The 4 diplomatic missions of Indonesia in Oceania by host country name
{'Australia': 'Australia', 'Fiji': 'Fiji', 'New Zealand': 'New Zealand', 'Papua New Guinea': 'Papua New Guinea'}
Largest known prime number > History
The 10 largest known prime numbers discovered by GIMPS by number of digits
Extraction Error: list index out of range
Comparison of mobile CRM systems > General
The 5 most recently updated mobile CRM systems by last updated year
Extraction Error: list index out of range
List of speech recognition software > Acoustic models and speech corpus (compilation)
The 5 open-source speech recognition applications by application name
{'CMU Sphinx': 'Yes', 'Julius': 'Yes', 'Kaldi': 'Yes', '

In [83]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import get_sampling_prompt

print(get_sampling_prompt(list(my_tables[0].values())[0], list(my_tables[0].values())[1]))

Analyze the Wikipedia table and identify a group of 3-10 entities.

    ### PRECISION RULES:
    1. THE GROUP NAME: Must be specific. Do NOT say 'Recent movies'. Say 'The 7 most recent movies' or 'Movies released 2010-2015'. 
    2. QUANTITY: Always include the count or the specific range bounds in the description.
    3. CONSTRAINTS: The result MUST contain at least 3 and no more than 10 rows.
    
    ### AVAILABLE STRATEGIES (Choose the best fit for this data):
    - [Numerical Strategy]: Threshold Filtering: Isolate 3-10 entities that fall above or below a specific significant threshold.
    - [Nested/List Strategy]: Complexity Count: Filter by the 'volume' of items in a cell (e.g., entities that have more than 5 but fewer than 10 associated countries/affiliates).
    - [Temporal Strategy]: Cutoff Logic: Filter for 3-10 entities that occurred specifically before or after a major milestone year.
    - [Categorical Strategy]: Multi-Column Pivot: Combine two categorical columns (e.g.,

In [112]:
import re

def smart_profile_table(table_rows):
    if not table_rows:
        return {}
    
    num_rows = len(table_rows)
    keys = list(table_rows[0].keys())
    profile = {"temporal": [], "numerical": [], "list": []}
    
    # Semantic keywords for temporal data
    temporal_keywords = ['year', 'date', 'founded', 'since', 'introduced', 'created', 'period', 'time', 'opened', 'released']
    
    for key in keys:
        col_name = str(key).lower()
        values = [str(row.get(key, "")).strip() for row in table_rows if row.get(key)]
        if not values: continue
        
        unique_count = len(set(values))
        
        # 1. SMART TEMPORAL CHECK
        # Must have a temporal keyword in the header OR match a strict year pattern
        has_temporal_name = any(word in col_name for word in temporal_keywords)
        # Check if values look like years/dates (e.g., 1995, 2024, or 12/05/2010)
        is_year_pattern = sum(1 for v in values if re.search(r'\b(1[789]\d{2}|20\d{2})\b', v)) / num_rows > 0.6
        
        if has_temporal_name or is_year_pattern:
            profile["temporal"].append({"col": key, "unique": unique_count})
            continue

        # 2. SMART NUMERICAL CHECK
        # Must be "mostly digits" for the majority of rows. 
        # We strip common symbols but expect a numeric core.
        numeric_count = 0
        for v in values:
            # Remove currency, commas, and units (km, mi, %)
            clean = re.sub(r'[^\d.]', '', v)
            if clean and clean.replace('.', '', 1).isdigit():
                numeric_count += 1
        
        if numeric_count / num_rows > 0.7:
            profile["numerical"].append({"col": key, "unique": unique_count})
            continue

        # 3. CATEGORICAL LIST CHECK
        # If it wasn't temporal or numerical, check if it's a repeating category.
        # Ratio check: Unique values should be between 3 and 80% of the total rows.
        if 3 <= unique_count <= (num_rows * 0.8):
            profile["list"].append({"col": key, "unique": unique_count})
            
    return profile

In [120]:
t = random.choice(my_tables)

smart_profile_table(t['rows'])

{'temporal': [{'col': 'Year', 'unique': 6}],
 'numerical': [{'col': 'Listicle', 'unique': 11}],
 'list': [{'col': 'Placement', 'unique': 7}]}

In [121]:
t

{'table_name': 'List of awards and nominations received by G-Dragon > Other accolades > Listicles',
 'rows': [{'Publisher': 'Arena Korea',
   'Year': 2008,
   'Listicle': 'Most Influential Men of the Year',
   'Placement': '1st'},
  {'Publisher': 'Business of Fashion',
   'Year': 2015,
   'Listicle': 'BoF 500',
   'Placement': 'Included'},
  {'Publisher': 'Complex',
   'Year': 2025,
   'Listicle': 'The 25 Best Dressed Celebs of the 21st Century',
   'Placement': '16th'},
  {'Publisher': 'Forbes',
   'Year': 2012,
   'Listicle': '2030 Power Leaders List',
   'Placement': 'Included'},
  {'Publisher': 'Gallup Korea',
   'Year': 2013,
   'Listicle': 'Top 10 Singers of the Year',
   'Placement': '10th'},
  {'Publisher': 'Golden Disc Awards',
   'Year': 2025,
   'Listicle': 'Golden Disc Powerhouse 40',
   'Placement': 'Included'},
  {'Publisher': 'GQ Korea',
   'Year': 2015,
   'Listicle': 'Men of the Year for 2015',
   'Placement': '1st'},
  {'Publisher': 'The Guardian',
   'Year': 2018,
  

In [5]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import process_small_tables_batch

small_table_sample = random.sample([t for t in final_data if len(t.get('rows', [])) <= 10], 5)
# await process_small_tables_batch(small_table_sample)

In [6]:
small_table_sample

[{'table_name': "List of teams and cyclists in the 2024 Giro d'Italia Women > Cyclists > By team",
  'rows': [{'No.': 41, 'Rider': 'Cédrine Kerbaol\xa0(FRA)', 'Pos.': 'DNF-4'},
   {'No.': 42, 'Rider': 'Alice Maria Arzuffi\xa0(ITA)', 'Pos.': 17},
   {'No.': 43, 'Rider': 'Laura Asencio\xa0(FRA)', 'Pos.': 31},
   {'No.': 44, 'Rider': 'Mylène de Zoete\xa0(NED)', 'Pos.': 99},
   {'No.': 45, 'Rider': 'Arianna Fidanza\xa0(ITA)', 'Pos.': 82},
   {'No.': 46, 'Rider': 'Marta Jaskulska\xa0(POL)', 'Pos.': 81},
   {'No.': 47, 'Rider': 'Kathrin Schweinberger\xa0(AUT)', 'Pos.': 92}]},
 {'table_name': "List of teams and cyclists in the 2024 Giro d'Italia Women > Cyclists > By team",
  'rows': [{'No.': 81, 'Rider': 'Pauliena Rooijakkers\xa0(NED)', 'Pos.': 4},
   {'No.': 82, 'Rider': 'Aniek van Alphen\xa0(NED)', 'Pos.': 76},
   {'No.': 83, 'Rider': 'Ceylin del Carmen Alvarado\xa0(NED)', 'Pos.': 58},
   {'No.': 84, 'Rider': 'Sanne Cant\xa0(BEL)', 'Pos.': 37},
   {'No.': 85, 'Rider': 'Millie Couzens\xa0(G

In [7]:
await process_small_tables_batch(small_table_sample[:3])

Usage cost: 0.0001$ (Estimated: 0.00$):  33%|██████████████▎                            | 1/3 [05:04<10:09, 304.88s/it]


CancelledError: 

In [23]:
for i in range(3, 11):
    for t in final_data:
        if len(t['rows']) == i:
            if 'List of countries' in t['table_name']:
                print(t['table_name'])
                print("="*100)
                break

List of countries that have the same national anthem > Countries that used to share the same melody
List of countries by diamond exports > Sources
List of countries without an airport > Sovereign states
List of countries that have gained independence from the United Kingdom > Evolution of dominions to independence
List of countries and dependencies and their capitals in native languages > F
List of countries and dependencies and their capitals in native languages > K
List of countries and dependencies and their capitals in native languages > E
List of countries by meat production > World production


In [25]:
for t in final_data:
    if len(t['rows']) <= 10:
        if 'List of countries' in t['table_name']:
            print(t['table_name'])

List of countries by diamond exports > Sources
List of countries by meat production > World production
List of countries by meat production > World production
List of countries by meat production > World production
List of countries by meat production > World production
List of countries by meat production > World production
List of countries by meat production > World production
List of countries that have gained independence from the United Kingdom > Evolution of dominions to independence
List of countries and dependencies and their capitals in native languages > D
List of countries and dependencies and their capitals in native languages > E
List of countries and dependencies and their capitals in native languages > F
List of countries and dependencies and their capitals in native languages > H
List of countries and dependencies and their capitals in native languages > I
List of countries and dependencies and their capitals in native languages > J
List of countries and dependencies a

In [26]:
import networkx as nx
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def clean_title(title):
    """Normalizes titles to increase clustering accuracy."""
    # Remove 'List of', 'Table of', punctuation, and extra whitespace
    t = title.lower()
    t = re.sub(r'\b(list of|table of|category:)\b', '', t)
    t = re.sub(r'[^\w\s]', ' ', t)
    return " ".join(t.split())

def cluster_and_deduplicate_tables(tables, similarity_threshold=0.85):
    """
    Groups tables by title similarity and returns one table per cluster.
    Input: List of dicts [{'table_name': '...', 'rows': [...]}, ...]
    """
    if not tables:
        return []

    # 1. Prepare Normalized Titles
    original_titles = [t.get('table_name', '') for t in tables]
    cleaned_titles = [clean_title(ttl) for ttl in original_titles]

    # 2. Vectorize using Character N-Grams (robust against typos/plurals)
    vectorizer = TfidfVectorizer(ngram_range=(2, 4), analyzer='char_wb')
    tfidf_matrix = vectorizer.fit_transform(cleaned_titles)

    # 3. Compute Sparse Similarity
    # Using linear_kernel or dot product is faster for normalized TF-IDF
    sim_matrix = cosine_similarity(tfidf_matrix)
    
    # 4. Build the Graph
    G = nx.Graph()
    G.add_nodes_from(range(len(tables)))
    
    # Efficiently find indices above threshold
    rows, cols = np.where(sim_matrix > similarity_threshold)
    
    # Add edges (r < c ensures we don't add self-loops or double edges)
    edges = [(r, c) for r, c in zip(rows, cols) if r < c]
    G.add_edges_from(edges)

    # 5. Extract Clusters and Select Representatives
    final_tables = []
    clusters = list(nx.connected_components(G))
    
    for cluster_indices in clusters:
        # Strategy: Pick the table with the most rows as the cluster representative
        # This ensures you keep the 'richest' data source for that topic
        best_index = max(cluster_indices, key=lambda i: len(tables[i].get('rows', [])))
        final_tables.append(tables[best_index])

    return final_tables

In [28]:
# Assuming 'all_tables' is your list of 30k dictionaries
deduped_dataset = cluster_and_deduplicate_tables(final_data, similarity_threshold=0.8)

print(f"Original: {len(final_data)} | Deduped: {len(deduped_dataset)}")

Original: 7104 | Deduped: 3378


In [29]:
for t in deduped_dataset:
    if len(t['rows']) <= 10:
        if 'List of countries' in t['table_name']:
            print(t['table_name'])

List of countries by diamond exports > Sources
List of countries that have gained independence from the United Kingdom > Evolution of dominions to independence
List of countries that have the same national anthem > Countries that share the same melody
List of countries without an airport > Sovereign states
List of countries without an airport > Non-self-governing territories
List of countries and territories by number of land borders > Artificial border crossings
List of countries that have gained independence from Portugal > Abandoned territories
List of countries by energy intensity > Sources


In [34]:
short = 0 #[0, 0, 0, 0, 0, 0, 0, 0]
midd = 0
long = 0
for l in [len(sample['rows']) for sample in deduped_dataset]:
    if l > 25:
        long += 1
    elif l > 12:
        midd += 1
    else:
        short += 1
        # short[l - 3] += 1

print(f"Short: {short}, Mid: {midd}, Long: {long}")
print("=" * 100)

for i in range(3, 13):
    for t in deduped_dataset:
        if len(t['rows']) == i:
            print(t['table_name'])
            break

Short: 1787, Mid: 799, Long: 792
Comparison of OS emulation or virtualization apps on Android > Terminal emulation apps utilizing internal OS
ECHL awards > Team trophies
51st Primetime Emmy Awards > Most major nominations
List of Welsh flags > Historical > Religious flags
2011 ESPY Awards > Categories
List of Tennessee Titans head coaches > Key
List of Welsh flags > Regions, counties and cities > Traditional counties
ECHL awards > Individual trophies and awards
List of motocross riders > World Championships by nationality
List of Indian Premier League awards > Best catch
